## Домашнее задание 1

## Установка зависимостей

> **Решение сдаётся zip-архивом: notebook.ipynb + файлы trace1 ... traceN.nc** (File → Export project as ZIP)

In [ ]:
%pip install -U pymc numpy scipy

## Оценка покупок

In [ ]:
import pymc as pm
import pandas as pd
import io
import pytensor.tensor as pt
import matplotlib.pyplot as plt
import arviz as az
%matplotlib inline

У вас есть сырые данные о покупках (**obs**) одного и того же товара в трёх продуктовых магазинах (**store**) (ноль означает, что покупатель не купил товар). Магазины разного размера, поэтому число наблюдений по ним различается. Нужно сравнить, какой магазин эффективнее с точки зрения продаж.


1. (5 баллов) Предложите и обоснуйте функцию правдоподобия для задачи
    * правдоподобие с отрицательным носителем = -5 баллов
2. (10 баллов) Предложите и обоснуйте вероятностную модель (неиерархическую) для задачи. При желании часть параметров можно сделать общими, а часть — независимыми.
    0. (2.5 балла) Запишите модель в LaTeX (допустимо просто отобразить модель)
    1. (2.5 балла) Обоснуйте выбор априорных распределений, исходя только из предположения, что покупки имеют порядок сотен и не превышают 500
    2. (2.5 балла) Проведите априорную предсказательную проверку (prior predictive check), чтобы подтвердить свой выбор
    3. (2.5 балла) Сэмплируйте из модели, постройте графики трейсов, сравните их с априорными распределениями и прокомментируйте (например, влияние объёма данных).
    4. **итоговый балл за раздел умножается на (1 - число дивергентных сэмплов / общее число сэмплов)**
    5. Сохраните трейс в traceN.nc
3. (10 баллов) Предложите иерархическую модель для задачи. Введите иерархию только для одного параметра модели (выбор за вами, предпочтительно — среднее) аналогично п. 2.

4. (10 баллов) Предложите иерархическую модель для задачи. Введите иерархию для 2 параметров модели (второй параметр выберите сами, предпочтительно — p) аналогично п. 2

5. (5 баллов) Проведите сравнение моделей (см. [`az.compare`](https://arviz-devs.github.io/arviz/api/generated/arviz.compare.html)), можно использовать уже построенные модели (не менее 2)
    1. Интерпретируйте результат и выберите модель, разберите предупреждения (если они возникают при построении моделей или результатов)
6. (5 баллов) Бизнес-метрика (бонус)
    1. Предложите и обоснуйте метрику для сравнения магазинов, позволяющую сказать «A лучше, чем B»
    2. Посчитайте эту метрику на лучшей из ваших моделей
7. (5 баллов) Сделайте прогноз для нового магазина, для этого понадобится модель из п. 4 (бонус)

(максимум 40 баллов)

**Важно**
> Оформляйте ключевые моменты решения в виде текстовых ячеек

In [ ]:
raw_data = io.StringIO('obs,store\n117,0\n0,0\n0,0\n134,0\n0,0\n0,0\n0,0\n0,0\n122,0\n0,0\n108,0\n0,0\n0,0\n0,0\n0,0\n124,0\n0,0\n0,0\n113,0\n118,0\n0,0\n105,0\n114,0\n0,0\n118,0\n106,0\n0,0\n100,0\n0,0\n0,0\n123,0\n0,0\n0,0\n0,0\n0,0\n129,0\n0,0\n117,0\n0,0\n0,0\n0,0\n0,0\n0,0\n115,0\n123,0\n0,0\n0,0\n0,0\n137,0\n0,0\n0,0\n0,0\n0,0\n0,0\n136,0\n0,0\n107,0\n0,0\n0,0\n0,0\n0,0\n115,0\n118,0\n0,0\n105,0\n105,0\n97,0\n121,0\n0,0\n0,0\n0,0\n124,0\n114,0\n0,0\n101,0\n0,0\n0,0\n0,0\n121,0\n0,0\n0,0\n0,0\n119,0\n0,0\n108,0\n121,0\n0,0\n0,0\n116,0\n0,0\n0,0\n0,0\n0,0\n0,0\n0,0\n0,0\n127,0\n0,0\n0,0\n106,0\n0,0\n0,0\n0,0\n131,0\n0,0\n0,0\n128,0\n122,0\n0,0\n0,0\n0,0\n0,0\n0,0\n0,0\n102,0\n119,0\n0,0\n0,0\n0,0\n123,0\n0,0\n0,0\n0,0\n120,0\n0,1\n0,1\n0,1\n0,1\n0,1\n121,1\n0,1\n0,1\n0,1\n123,1\n0,1\n106,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n142,1\n0,1\n0,1\n0,1\n131,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,1\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n81,2\n0,2\n0,2\n73,2\n0,2\n0,2\n0,2\n69,2\n0,2\n94,2\n0,2\n91,2\n0,2\n0,2\n98,2\n0,2\n79,2\n90,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n102,2\n0,2\n87,2\n0,2\n81,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n86,2\n97,2\n0,2\n0,2\n0,2\n76,2\n0,2\n83,2\n0,2\n0,2\n88,2\n0,2\n86,2\n0,2\n0,2\n99,2\n0,2\n0,2\n86,2\n86,2\n0,2\n93,2\n0,2\n0,2\n0,2\n90,2\n0,2\n81,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n91,2\n0,2\n0,2\n67,2\n107,2\n0,2\n0,2\n0,2\n0,2\n101,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n75,2\n112,2\n0,2\n0,2\n0,2\n0,2\n0,2\n91,2\n0,2\n0,2\n0,2\n81,2\n108,2\n92,2\n0,2\n0,2\n0,2\n90,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n93,2\n0,2\n0,2\n0,2\n96,2\n0,2\n0,2\n0,2\n0,2\n94,2\n0,2\n90,2\n0,2\n0,2\n0,2\n0,2\n94,2\n0,2\n0,2\n0,2\n0,2\n72,2\n0,2\n97,2\n0,2\n82,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n105,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n99,2\n90,2\n0,2\n76,2\n92,2\n0,2\n0,2\n0,2\n0,2\n0,2\n100,2\n113,2\n0,2\n0,2\n0,2\n100,2\n0,2\n91,2\n0,2\n88,2\n0,2\n0,2\n0,2\n83,2\n0,2\n0,2\n82,2\n0,2\n84,2\n101,2\n0,2\n0,2\n0,2\n0,2\n102,2\n0,2\n0,2\n0,2\n99,2\n91,2\n0,2\n95,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n73,2\n0,2\n80,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n90,2\n0,2\n0,2\n0,2\n0,2\n95,2\n0,2\n91,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n92,2\n0,2\n0,2\n97,2\n70,2\n102,2\n0,2\n0,2\n0,2\n87,2\n111,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n73,2\n79,2\n99,2\n0,2\n0,2\n88,2\n88,2\n0,2\n0,2\n94,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n94,2\n0,2\n88,2\n0,2\n0,2\n0,2\n0,2\n0,2\n80,2\n90,2\n0,2\n0,2\n0,2\n87,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n105,2\n0,2\n0,2\n88,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n71,2\n0,2\n103,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n96,2\n0,2\n0,2\n0,2\n94,2\n0,2\n79,2\n110,2\n0,2\n0,2\n0,2\n90,2\n0,2\n0,2\n0,2\n101,2\n0,2\n95,2\n94,2\n0,2\n0,2\n0,2\n0,2\n0,2\n85,2\n0,2\n100,2\n92,2\n0,2\n0,2\n0,2\n110,2\n0,2\n0,2\n0,2\n0,2\n78,2\n0,2\n74,2\n0,2\n103,2\n0,2\n0,2\n0,2\n0,2\n90,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n90,2\n105,2\n0,2\n0,2\n0,2\n103,2\n0,2\n0,2\n0,2\n0,2\n98,2\n89,2\n105,2\n0,2\n0,2\n93,2\n0,2\n93,2\n0,2\n0,2\n0,2\n0,2\n88,2\n88,2\n0,2\n0,2\n91,2\n0,2\n89,2\n0,2\n0,2\n100,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n75,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n85,2\n0,2\n83,2\n0,2\n0,2\n80,2\n0,2\n0,2\n0,2\n0,2\n89,2\n0,2\n0,2\n76,2\n0,2\n89,2\n0,2\n0,2\n0,2\n0,2\n0,2\n88,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n101,2\n0,2\n112,2\n92,2\n0,2\n0,2\n0,2\n0,2\n0,2\n93,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n0,2\n95,2\n0,2\n0,2\n94,2\n0,2\n0,2\n0,2\n0,2\n0,2\n104,2\n0,2\n89,2\n0,2\n88,2\n0,2\n0,2\n0,2\n0,2\n86,2\n82,2\n0,2\n0,2\n0,2\n0,2\n0,2\n91,2\n0,2\n83,2\n0,2\n81,2\n0,2\n0,2\n87,2\n0,2\n0,2\n0,2\n83,2\n0,2\n94,2\n0,2\n0,2\n81,2\n0,2\n0,2\n0,2\n87,2\n0,2\n0,2\n108,2\n')
data = pd.read_csv(raw_data)

In [ ]:
data

## 1. Описание данных
1. Предложите и обоснуйте функцию правдоподобия для задачи

## 2. Вероятностная модель

2. Предложите и обоснуйте вероятностную модель (неиерархическую) для задачи. При желании часть параметров можно сделать общими, а часть — независимыми.

    0. Запишите модель в LaTeX (допустимо просто отобразить модель в LaTeX)
    1. Обоснуйте выбор априорных распределений, исходя только из предположения, что покупки имеют порядок сотен и не превышают 500
    2. Проведите априорную предсказательную проверку (prior predictive check), чтобы подтвердить свой выбор
    3. Сэмплируйте из модели, постройте графики трейсов, сравните их с априорными распределениями и прокомментируйте (например, влияние объёма данных).

**подсказки:**

1. изучите `pt.stack`, чтобы справиться с проблемами размерностей у вероятностей компонент смеси (если используете смеси)
2. загляните в материалы второго семинара

In [ ]:
with pm.Model(coords=dict(store=range(3))) as model:
    ...

In [ ]:
with model:
    prior = pm.sample_prior_predictive()

In [ ]:
with model:
    trace = pm.sample()
    trace.extend(prior)

## 3. Иерархическая модель (1 параметр)
3. Предложите иерархическую модель для задачи. Введите иерархию только для одного параметра модели (выбор за вами, предпочтительно — среднее)

**подсказки:**

1. Используйте нецентрированную параметризацию
2. Используйте pm.Deterministic, чтобы отслеживать репараметризованные параметры

In [ ]:
with pm.Model(coords=dict(store=range(3))) as model1:
    ...

In [ ]:
with model1:
    prior1 = pm.sample_prior_predictive()

In [ ]:
with model1:
    trace1 = pm.sample()
    trace1.extend(prior1)

## 4. Иерархическая модель (2 параметра)
4. Предложите иерархическую модель для задачи. Введите иерархию для 2 параметров модели (второй параметр выберите сами, предпочтительно — p)

**подсказка:**

Для иерархии на вероятностях:

Используйте распределение LogitNormal (предпочтительно)

1. его легко интерпретировать
2. его проще параметризовать (так же, как нормальное распределение, только с преобразованием)

In [ ]:
with pm.Model(coords=dict(store=range(3))) as model2:
    ...

In [ ]:
with model2:
    prior2 = pm.sample_prior_predictive()

In [ ]:
with model2:
    trace2 = pm.sample()
    trace2.extend(prior2)

## 5. Сравнение моделей

5. Проведите сравнение моделей (см. az.compare)

    1. Интерпретируйте результат и выберите лучшую модель
    2. Разберите предупреждения, если они есть

In [ ]:
res = az.compare(dict(simple=trace, hierarchy1=trace1, hierarchy2=trace2))

In [ ]:
res